Pakiet torchtext składa się z narzędzi do przetwarzania danych i popularnych zestawów danych dla języka naturalnego.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import torch.nn.functional as F
import keras
from tqdm.notebook import trange, tqdm

## Tokenizacja tekstu
Tokenizacja tekstu jest pierwszym krokiem przygotowania tekstu do modelowania. Niestety, tokenizer z biblioteki torchtext nie jest łatwy do zastosowania ze względu na niezgodność wersji torcha i torchtext.

In [ ]:
'''from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
text = "I like Word of Tanks."
tokens = tokenizer(text)
print(tokens)'''

'from torchtext.data.utils import get_tokenizer\nfrom torchtext.vocab import build_vocab_from_iterator\ntokenizer = torchtext.data.utils.get_tokenizer(\'basic_english\')\ntext = "I like Word of Tanks."\ntokens = tokenizer(text)\nprint(tokens)'

In [ ]:
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.preprocessing.text import text_to_word_sequence
# definiowanie dokumentu
text = 'I like to play the game Word of Tanks'
# tokenizacja dokumentu
result = text_to_word_sequence(text)
print(result)

['i', 'like', 'to', 'play', 'the', 'game', 'word', 'of', 'tanks']


## Warstwy osadzania (Embedding Layers) w PyTorch

[Embedding Layers](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) to przydatna funkcja PyTorch, która umożliwia programowi automatyczne wstawianie dodatkowych informacji do przepływu danych sieci neuronowej. Warstwa osadzania automatycznie umożliwia wstawianie wektorów w miejsce indeksów słów.  

Programiści często używają warstw osadzania z przetwarzaniem języka naturalnego (NLP); można jednak użyć tych warstw, gdy chcesz wstawić dłuższy wektor w miejscu wartości indeksu. W pewnym sensie można myśleć o warstwie osadzania jako o rozszerzeniu wymiaru. Istnieje jednak nadzieja, że te dodatkowe wymiary dostarczą modelowi więcej informacji i zapewnią lepszy wynik.


## Przykład prostej warstwy osadzania

* **num_embeddings** = Jak duży jest słownik?  Ile kategorii kodujesz? Ten parametr to liczba elementów w „tabeli wyszukiwania”.
* **embedding_dim** = Ile liczb w wektorze chcesz zwrócić.

Teraz zbudujemy sieć neuronową ze słownikiem o rozmiarze 10, która zredukuje wartości od 0 do 9 do 4 wektorów liczb. Ta sieć neuronowa nie robi nic więcej poza przekazaniem osadzenia na wyjściu. Ale pozwala nam zobaczyć, co robi osadzanie. Każdy przychodzący wektor cech będzie miał dwie takie cechy.


In [ ]:
import torch
import torch.nn as nn

embedding_layer = nn.Embedding(num_embeddings=10, embedding_dim=4)
optimizer = torch.optim.Adam(embedding_layer.parameters(), lr=0.001)
loss_function = nn.MSELoss()

Zobaczmy, co otrzymujemy.

In [ ]:
print(embedding_layer)

Embedding(10, 4)


Dla tej sieci neuronowej, która jest tylko warstwą osadzania, wejściem jest wektor o rozmiarze 2. Te dwa wejścia są liczbami całkowitymi od 0 do 9 (odpowiadającymi żądanej liczbie input_dim wynoszącej 10 wartości). Patrząc na powyższe podsumowanie, widzimy, że warstwa osadzania ma 40 parametrów. Wartość ta pochodzi z wbudowanej tabeli odnośników, która zawiera cztery ilości (output_dim) dla każdej z 10 (input_dim) możliwych wartości całkowitych dla dwóch wejść. Wyjście to 2 wektory (input_length) o długości 4 (output_dim), co daje całkowity rozmiar wyjścia 8, co odpowiada kształtowi wyjścia podanemu w powyższym podsumowaniu.

Teraz przepuścimy te wiersze przez sieć neuronową. Dane wejściowe to dwie wartości całkowite, jak określono podczas tworzenia sieci neuronowej.


In [ ]:
input_tensor = torch.tensor([[1, 2]], dtype=torch.long)
pred = embedding_layer(input_tensor)

print(input_tensor.shape)
print(pred)


torch.Size([1, 2])
tensor([[[ 0.2418,  0.5576,  0.2102,  0.2638],
         [ 0.3323, -0.3347, -0.5191,  0.3677]]], grad_fn=<EmbeddingBackward0>)


Tutaj widzimy dwa wektory o długości 4, które PyTorch wyszukał dla każdej wejściowej liczby całkowitej. Przypomnijmy, że tablice Pythona są oparte na zerze. PyTorch zastąpił wartość 1 drugim wierszem macierzy wyszukiwania 10 x 4. Podobnie PyTorch zwrócił wartość 2 przez trzeci wiersz macierzy wyszukiwania. Poniższy kod wyświetla macierz wyszukiwania w całości. Warstwa osadzania nie wykonuje żadnych operacji matematycznych poza wstawieniem poprawnego wiersza z tabeli odnośników.

In [ ]:
embedding_layer.weight.data

tensor([[-2.0272, -0.1033,  2.6555, -0.2537],
        [ 0.2418,  0.5576,  0.2102,  0.2638],
        [ 0.3323, -0.3347, -0.5191,  0.3677],
        [ 1.7016,  1.3843,  0.3611,  0.4816],
        [ 0.5896,  1.0025, -1.7613, -0.3915],
        [-0.6974,  0.0770, -0.8253,  0.8431],
        [-1.0904,  0.2926, -0.5060,  1.4109],
        [-0.7998,  1.3403,  0.2279, -0.4626],
        [ 1.5860,  0.7623,  0.4996,  0.4558],
        [-0.1571,  1.1447,  0.9617, -1.2608]])

Powyższe wartości są losowymi parametrami, które PyTorch wygenerował jako punkty początkowe.  Ogólnie rzecz biorąc, przeniesiemy osadzenie lub wytrenujemy te losowe wartości w coś użytecznego.  Poniższa sekcja pokazuje, jak osadzić ręcznie zakodowane osadzenie.

## Przenoszenie osadzenia

Teraz zobaczymy, jak na sztywno zakodować wyszukiwanie osadzania, które wykonuje proste kodowanie one-hot.  Kodowanie one-hot przekształciłoby wejściowe wartości całkowite 0, 1 i 2 na wektory odpowiednio $[1,0,0]$, $[0,1,0]$ i $[0,0,1]$. Poniższy kod zastąpił losowe wartości wyszukiwania w warstwie osadzania tabelą wyszukiwania inspirowaną kodowaniem one-hot.

In [ ]:
import torch
import torch.nn as nn

# Define the embedding lookup matrix
embedding_lookup = torch.tensor([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=torch.float32)  # Make sure to use float32 for weight matrices

# Create the embedding layer
embedding_layer = nn.Embedding(num_embeddings=3, embedding_dim=3)

# Set the weights of the embedding layer
embedding_layer.weight.data = embedding_lookup


Mamy następujące parametry dla warstwy Embedding:
    
* input_dim=3 - Dozwolone są trzy różne całkowite wartości kategoryczne.
* output_dim=3 - Trzy kolumny reprezentują wartość kategoryczną z trzema możliwymi wartościami na kodowanie one-hot.
* input_length=2 - Wektor wejściowy ma dwie z tych wartości kategorycznych.

Wysyłamy zapytanie do sieci neuronowej z dwiema kategorycznymi wartościami values to see the lookup performed.

In [ ]:
# Create the input tensor directly in PyTorch
input_tensor = torch.tensor([[0, 1]], dtype=torch.long)

# Forward pass to get the predictions
pred = embedding_layer(input_tensor)

print(input_tensor.shape)
print(pred)

torch.Size([1, 2])
tensor([[[1., 0., 0.],
         [0., 1., 0.]]], grad_fn=<EmbeddingBackward0>)


Podane dane wyjściowe pokazują, że dostarczyliśmy programowi dwa wiersze z tabeli kodowania one-hot. To kodowanie jest poprawnym kodowaniem one-hot dla wartości 0 i 1, gdzie możliwe są maksymalnie 3 unikalne wartości.

Poniższa sekcja pokazuje, jak wytrenować tę tabelę wyszukiwania osadzania.

## Trenowanie osadzania

Po pierwsze, importujemy niezbędne bibiloteki.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import OneHotEncoder
from torch.nn.utils.rnn import pad_sequence

Tworzymy sieć neuronową, która klasyfikuje recenzje restauracji jako pozytywne lub negatywne. Ta sieć neuronowa może akceptować ciągi jako dane wejściowe, takie jak podane tutaj. Ten kod zawiera również pozytywne lub negatywne etykiety dla każdej recenzji.

In [ ]:
# Define 10 resturant reviews.
reviews = [
    'Never coming back!',
    'Horrible service',
    'Rude waitress',
    'Cold food.',
    'Horrible food!',
    'Awesome',
    'Awesome service!',
    'Rocks!',
    'poor work',
    'Couldn\'t have done better']

# Define labels (1=negative, 0=positive)
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

Zauważ, że przedostatnia etykieta jest nieprawidłowa.  Błędy takie jak ten nie są zbyt niezwykłe, ponieważ większość danych szkoleniowych może zawierać pewne zakłócenia.

Definiujemy rozmiar słownictwa na 50 słów.  Chociaż nie mamy 50 słów, dobrze jest użyć wartości większej niż potrzeba.  Jeśli jest więcej niż 50 słów, najrzadziej używane słowa w zestawie uczącym są automatycznie odrzucane przez warstwę osadzania podczas uczenia.  W przypadku danych wejściowych kodujemy ciągi za pomocą jednego skrótu.  Używamy tutaj metody kodowania one-hot TensorFlow, a nie Scikit-Learn. Scikit-learn rozszerzyłby te ciągi do 0 i 1, jak zwykle widzimy w przypadku zmiennych fikcyjnych.  TensorFlow tłumaczy wszystkie słowa na wartości indeksu i zastępuje każde słowo tym indeksem.

In [ ]:
# One-hot encode reviews
VOCAB_SIZE = 50
encoded_reviews = [torch.tensor([hash(word) % VOCAB_SIZE for word in review.split()]) for review in reviews]

print(f"Encoded reviews: {encoded_reviews}")

Encoded reviews: [tensor([17, 36,  2]), tensor([6, 2]), tensor([33, 33]), tensor([ 2, 37]), tensor([ 6, 48]), tensor([28]), tensor([28, 26]), tensor([2]), tensor([18, 10]), tensor([38, 21,  4, 35])]


Program one-hot koduje te recenzje do indeksów słów; jednak ich długości są różne.  Skracamy te recenzje do 4 słów i obcinamy wszystkie słowa wykraczające poza czwarte słowo.

In [ ]:
MAX_LENGTH = 4
padded_reviews = pad_sequence(encoded_reviews, batch_first=True, padding_value=0).narrow(1, 0, MAX_LENGTH)
print(padded_reviews)


tensor([[17, 36,  2,  0],
        [ 6,  2,  0,  0],
        [33, 33,  0,  0],
        [ 2, 37,  0,  0],
        [ 6, 48,  0,  0],
        [28,  0,  0,  0],
        [28, 26,  0,  0],
        [ 2,  0,  0,  0],
        [18, 10,  0,  0],
        [38, 21,  4, 35]])


Zgodnie z ustawieniem **padding=post**, każda recenzja jest uzupełniana przez dodanie zer na końcu, zgodnie z ustawieniem **padding=post**.

Następnie tworzymy sieć neuronową, która uczy się klasyfikować te recenzje.

In [ ]:
model = nn.Sequential(
    nn.Embedding(VOCAB_SIZE, 8),
    nn.Flatten(),
    nn.Linear(8 * MAX_LENGTH, 1),
    nn.Sigmoid()
)

Ta sieć akceptuje cztery wejścia całkowite, które określają indeksy wyściełanej recenzji filmu. Pierwsza warstwa osadzania konwertuje te cztery indeksy na cztery wektory długości 8. Wektory te pochodzą z tabeli wyszukiwania, która zawiera 50 (VOCAB_SIZE) wierszy wektorów o długości 8. Kodowanie to jest widoczne w 400 (8 razy 50) parametrach w warstwie osadzania. Rozmiar wyjścia z warstwy osadzania wynosi 32 (4 słowa wyrażone jako 8-cyfrowe osadzone wektory). Pojedynczy neuron wyjściowy jest połączony z warstwą osadzania za pomocą 33 wag (32 z warstwy osadzania i pojedynczy neuron bias). Ponieważ jest to jednoklasowa sieć klasyfikacyjna, używamy sigmoidalnej funkcji aktywacji i binary_crossentropy.

Program trenuje teraz sieć neuronową. Weryfikacja osadzenia i 33 wagi są aktualizowane w celu uzyskania lepszego wyniku.

In [ ]:
criterion = nn.BCELoss()  # Binary Cross Entropy
optimizer = optim.Adam(model.parameters())

# Training the model
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(padded_reviews.long())
    loss = criterion(outputs.squeeze(), torch.tensor(labels, dtype=torch.float))
    loss.backward()
    optimizer.step()

Możemy zobaczyć wyuczone osadzenia.  Potraktujmy wektor każdego słowa jako lokalizację w 8-wymiarowej przestrzeni, w której słowa związane z pozytywnymi recenzjami znajdują się blisko innych słów.  Podobnie, trening umieszcza negatywne recenzje blisko siebie.  Oprócz treningu ustawiającego te osadzenia, 33 wagi między warstwą osadzania a neuronem wyjściowym podobnie uczą się przekształcać te osadzenia w rzeczywistą prognozę.  Te osadzenia można zobaczyć tutaj.

In [ ]:
embedding_weights = list(model[0].parameters())[0]
print(embedding_weights.shape)
print(embedding_weights)

torch.Size([50, 8])
Parameter containing:
tensor([[-0.5056,  1.0850, -1.7348,  1.5662, -0.2990, -0.1452, -0.6516,  1.0402],
        [-0.3876,  1.0600,  0.4185,  2.0797,  2.2885, -2.0745, -0.2961, -0.5270],
        [-2.0661,  0.0034, -1.5175, -0.4361, -1.2034, -1.7031,  1.6595,  1.4387],
        [-1.8415, -1.7323,  0.5847, -0.1343, -0.4888, -1.6740,  2.7872, -1.7410],
        [ 0.8193, -0.1300, -1.5096, -0.3345, -0.6166,  1.4057,  0.7657,  0.4554],
        [-0.5809, -0.7134,  0.8018,  0.7103,  0.6998, -0.6703,  0.3472,  0.1891],
        [ 0.5733,  0.0404,  0.8909,  0.6104, -1.6747, -0.4114, -0.1082,  0.4840],
        [ 0.2499,  0.0355, -1.8921,  1.0118,  2.1640,  2.6242,  1.2267,  0.2116],
        [-0.5872, -1.0533,  0.1683,  1.7609, -0.1140,  1.0095, -0.4700,  0.9282],
        [-0.0119,  0.7768, -0.3005, -0.5830, -0.0846, -0.8683,  0.6073, -0.0146],
        [ 1.0697, -1.7072,  0.6639,  0.2480, -1.9551, -0.5636,  0.5219, -0.9695],
        [-0.1026, -0.1309,  0.3097,  0.6455, -0.2538, -0

Możemy teraz ocenić dokładność tej sieci neuronowej, w tym osadzenia i wyuczonej gęstej warstwy.  

In [ ]:
# Evaluation
with torch.no_grad():
    outputs = model(padded_reviews.long())
    predictions = (outputs > 0.5).float().squeeze()
    accuracy = (predictions == torch.tensor(labels)).float().mean().item()
    loss_value = criterion(outputs.squeeze(), torch.tensor(labels, dtype=torch.float)).item()

print(f'Accuracy: {accuracy}')
print(f'Log-loss: {loss_value}')

Accuracy: 1.0
Log-loss: 0.4700518548488617


Dokładność jest świetna, ale może wystąpić nadmierne dopasowanie. Dobrze byłoby zastosować wczesne zatrzymanie, aby uniknąć nadmiernego dopasowania w przypadku bardziej złożonego zestawu danych. Jednak strata nie jest idealna. Mimo że przewidywane prawdopodobieństwa wskazywały na poprawną prognozę w każdym przypadku, program nie osiągnął absolutnej pewności co do każdej poprawnej odpowiedzi. Brak pewności był prawdopodobnie spowodowany niewielką ilością szumu (omówionego wcześniej) w zestawie danych. Niektóre słowa, które pojawiły się zarówno w pozytywnych, jak i negatywnych recenzjach, przyczyniły się do braku absolutnej pewności.